In [1]:
import numpy as np
from bstrapping.weights.gaussian_weights import GaussianWeights

from bstrapping.bootstrap_procedures.weighted_bootstrap import WeightedBootstrap
from bstrapping.synthetic_time_series.moving_average import MovingAverage
from bstrapping.weights.auto_regressive_weights import AutoRegressiveWeights
from bstrapping.weights.moving_average import MovingAverageWeights
import pandas as pd
import matplotlib as mpl
# Use the pgf backend (must be set before pyplot imported)
mpl.use('pgf')

import matplotlib.pyplot as plt
import scienceplots

from typing import List

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
mean = 4  # mean of the time series

# Names of the stochastic processes
name_dependence_coefficients = [
    "iid",
    "MA(1)",
    "MA(2)",
]

# Dependence coefficients of the stochastic processes, i.e. of the moving average processes
dependence_coefficients = [
    np.array([0]),
    np.array([0.5]),
    np.array([0.5 ** i for i in range(1, 3)]),
]



In [4]:
# List of names of the weights used in the respective multiplier bootstrap
list_name_weights = ['AR',
                     'Multiplier',
                     #'MA',
                     ]


# Evaluate different bootstrap procedures for different sample sizes and different MA stochastic processes
def evaluate_bootstraps(time_series: MovingAverage,
                        sample_size: int,
                        runs: int = 100,
                        alpha: float = 0.05):
    bootstrapped_variances = [[], [], []]
    mean_in_confidence_interval = [[], [], []]
    for _ in range(runs):
        samples = time_series.generate_samples(number_samples=sample_size)
        for i, weights in enumerate([AutoRegressiveWeights(samples=samples),
                                     GaussianWeights(samples=samples),
                                     #MovingAverageWeights(samples=samples)
                                     ]):
            bootstrap = WeightedBootstrap(samples=samples, weights=weights, number_bootstrap_samples=100)
            # asymptotic variance and confidence interval
            bootstrapped_variances[i].append(sample_size * bootstrap.bootstrapped_variance,
                                             )
            mean_in_confidence_interval[i].append(bootstrap.bootstrapped_confidence_interval(alpha=alpha)[0] <= mean <=
                                                  bootstrap.bootstrapped_confidence_interval(alpha=alpha)[1])
    return bootstrapped_variances, mean_in_confidence_interval

# Statistical evaluation of the bootstrap procedures for increasing sample size and dependencies

In [5]:
def full_benchmark(sample_sizes: List[int],
                   dependence_coefficients: List[np.ndarray],
                   names_dependence_coefficients: List[str],
                   runs: int = 100,
                   alpha: float = 0.05):
    benchmark = []
    for sample_size in sample_sizes:
        for index, dependence_coefficient in enumerate(dependence_coefficients):
            time_series_current = MovingAverage(mean=mean, parameters=dependence_coefficient)
            bootstrapped_variances, mean_in_confidence_interval = evaluate_bootstraps(sample_size=sample_size,
                                                                                      alpha=alpha,
                                                                                      runs=runs,
                                                                                      time_series=time_series_current)
            bootstrapped_variances = bootstrapped_variances[:-1]  # TODO: remove this later
            mean_in_confidence_interval = mean_in_confidence_interval[:-1]  # TODO: remove this later

            benchmark.append([time_series_current.asymptotic_variance,
                              time_series_current.variance,
                              names_dependence_coefficients[index],
                              sample_size] +
                             np.mean(bootstrapped_variances, axis=1).tolist()
                             + np.std(bootstrapped_variances, axis=1).tolist()
                             + [1 - alpha]
                             + (np.sum(mean_in_confidence_interval, axis=1) / runs).tolist()
                             )

    return (pd.DataFrame(benchmark, columns=pd.MultiIndex.from_tuples([("mean", "Asymptotic variance"),
                                                                       ("Variance", ""),
                                                                       ("Stochastic process", ""),
                                                                       ("Sample size", "")] +
                                                                      [("mean", name,) for name in list_name_weights] +
                                                                      [("std", name,) for name in list_name_weights] +
                                                                      [("In confidence interval", "Confidence level")]
                                                                      +
                                                                      [("In confidence interval", name,) for name in
                                                                       list_name_weights]
                                                                      )).set_index(["Sample size"]))


In [6]:
# %%capture
benchmark = full_benchmark(sample_sizes=[100],
                           dependence_coefficients=dependence_coefficients,
                           names_dependence_coefficients=name_dependence_coefficients,
                           runs=1)

100 samples with dimension 1 were obtained. 

Bootstrapping...


100%|██████████| 100/100 [00:00<00:00, 2517.38it/s]


100 samples with dimension 1 were obtained. 

Bootstrapping...


100%|██████████| 100/100 [00:00<00:00, 22133.53it/s]


100 samples with dimension 1 were obtained. 

Bootstrapping...


...

100%|██████████| 100/100 [00:00<00:00, 23250.02it/s]


100 samples with dimension 1 were obtained. 

Bootstrapping...


100%|██████████| 100/100 [00:00<00:00, 2690.35it/s]


100 samples with dimension 1 were obtained. 

Bootstrapping...


100%|██████████| 100/100 [00:00<00:00, 22281.68it/s]


In [7]:
benchmark

mean Variance Stochastic process      mean  \
            Asymptotic variance                                    AR   
Sample size                                                             
100                      1.0000   1.0000                iid  1.162855   
100                      2.2500   1.2500              MA(1)  2.375909   
100                      3.0625   1.3125              MA(2)  2.421506   

                        std            In confidence interval                  
            Multiplier   AR Multiplier       Confidence level   AR Multiplier  
Sample size                                                                    
100           1.136884  0.0        0.0                   0.95  1.0        1.0  
100           1.329404  0.0        0.0                   0.95  1.0        1.0  
100           1.533314  0.0        0.0                   0.95  1.0        1.0

In [8]:
plt.style.use(['science', 'ieee'])
plt.rcParams.update({'font.size': 15})
fig, a = plt.subplots(2, len(name_dependence_coefficients), figsize=(18, 12))

for index, name_dependence_coefficient in enumerate(name_dependence_coefficients):
    benchmark_wrt_dependence_coefficients = benchmark[benchmark["Stochastic process"] == name_dependence_coefficient]
    benchmark_wrt_dependence_coefficients["mean"].plot(yerr=benchmark_wrt_dependence_coefficients["std"],
                                                       xlabel="",
                                                       sharex=False,
                                                       legend=False,
                                                       #fontsize=15,
                                                       #ylabel="Bootstrapped variance",
                                                       ax=a[0][index])
    benchmark_wrt_dependence_coefficients["In confidence interval"].plot(xlabel="",
                                                                         #ylabel="Coverage probability", 
                                                                         legend=False,
                                                                         ax=a[1][index],
                                                                         #fontsize=15,
                                                                         ylim=[0, 1])

    a[0][index].set_title(name_dependence_coefficient,
                          #fontsize=15
                          )

a[0][0].set_ylabel("Bootstrapped variance")
a[1][int(len(name_dependence_coefficients) / 2)].set_xlabel("Sample size")
a[1][0].set_ylabel("Coverage probability")
a[0][-1].legend(["Baseline"] + list_name_weights)

plt.savefig('bootstrap_benchmark.pgf', format='pgf')
#plt.tight_layout()

In [9]:
# 1000,2000,5000 0,1,2

# 10000,20000,500000 30,50,70

# 1000 runs!